In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../market_data.db')

Rolling 20 day Average

In [15]:
query = """SELECT 
    date,
    ticker,
    close,
    AVG(close) OVER (
        PARTITION BY ticker 
        ORDER BY date 
        ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
    ) AS rolling_avg
FROM prices
ORDER BY ticker, date
LIMIT 60
"""
df = pd.read_sql(query, conn)
df

,date,ticker,close,rolling_avg
0,2021-03-26,ALHC,17.309999,17.309999
1,2021-03-29,ALHC,19.000000,18.155000
2,2021-03-30,ALHC,19.940001,18.750000
3,2021-03-31,ALHC,21.930000,19.545000
4,2021-04-01,ALHC,23.200001,20.276000
5,2021-04-05,ALHC,23.520000,20.816667
6,2021-04-06,ALHC,23.590000,21.212857
7,2021-04-07,ALHC,21.770000,21.282500
8,2021-04-08,ALHC,22.770000,21.447778
9,2021-04-09,ALHC,23.830000,21.686000


Daily returns + Rank

In [11]:
query = """
WITH returns AS (
    SELECT 
        date,
        ticker,
        close,
        (close - LAG(close) OVER (PARTITION BY ticker ORDER BY date)) 
            / LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS daily_return
    FROM prices
)
SELECT 
    date,
    ticker,
    daily_return,
    RANK() OVER (PARTITION BY date ORDER BY daily_return DESC) AS return_rank
FROM returns
WHERE date = (SELECT MAX(date) FROM prices)
ORDER BY return_rank
"""
df = pd.read_sql(query, conn)
df


,date,ticker,daily_return,return_rank
0,2026-09-25,ALHC,0.070404,1
1,2026-09-25,DDOG,0.043632,2
2,2026-09-25,JPM,0.013292,3
3,2026-09-25,NVS,0.013164,4
4,2026-09-25,KKR,0.010770,5
5,2026-09-25,AXP,0.010567,6
6,2026-09-25,BX,0.010323,7
7,2026-09-25,WFC,0.009613,8
8,2026-09-25,SNOW,0.005808,9
9,2026-09-25,BIIB,0.005478,10
